## Setup

In [0]:
pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.autolog()

## Get the conf from the local conf file
model_config = mlflow.models.ModelConfig(development_config="../conf/chapter05_conf.yml")

retriever_configs = model_config.get("retriever_configs")

## Single Retriever

In [0]:
from typing import List, Dict, Optional, Any
from databricks.vector_search.client import VectorSearchClient

from mlflow.entities import SpanType, Document


class VectorSearchWrapper:
    def __init__(self, retriever_config: Dict):
        """
        Initialize the VectorSearchWrapper with a single retriever config.

        :param retriever_config: Dictionary containing the retriever config.
            Example:
            {
              'endpoint_name': 'vs_endpoint',
              'index_name': 'workspace.unity_air.faq_index',
              'columns': ['id', 'question', 'answer', 'search_text'],
              'k': 3,
              'retriever_schema': {
                  'primary_key': 'id',
                  'text_column': 'search_text',
                  'doc_uri': 'id',
                  'name': 'unity_air_faq_vs_index'
              }
            }
        """
        self.vsc = VectorSearchClient()
        self.retriever_cfg = retriever_config

        # Create the index handle
        self.index = self.vsc.get_index(
            endpoint_name=retriever_config["endpoint_name"],
            index_name=retriever_config["index_name"],
        )
    
    @mlflow.trace(span_type=SpanType.RETRIEVER, name="single_retriever_search", attributes={"vs_type": "databricks_vector_search"})
    def search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ):
        """
        Perform a similarity search against the retriever's index.

        :param query_text: The text query to search for.
        :param columns: Optional override for columns to return.
        :param filters: Optional filters to apply.
        :param num_results: Optional override for number of results (k).
        :return: Search results from the vector index.
        """
        
        mlflow.update_current_trace(tags={"vs_endpoint_name": self.retriever_cfg["endpoint_name"]})
        mlflow.update_current_trace(tags={"vs_index_name": self.retriever_cfg["index_name"]})

        span = mlflow.get_current_active_span()
        span.set_attribute("filters", filters or self.retriever_cfg.get("filters", {}))
        span.set_attribute("retriever_k", num_results or self.retriever_cfg.get("k", 5))
                             
        return self.index.similarity_search(
            query_text=query_text,
            columns=columns or self.retriever_cfg.get("columns", []),
            filters=filters or self.retriever_cfg.get("filters", {}),
            num_results=num_results or self.retriever_cfg.get("k", 5),
        )
    
    @mlflow.trace(span_type=SpanType.PARSER, name="parse_search_result")
    def parse_search_result(self, search_result):
        columns = search_result['manifest']["columns"]
        data_array = search_result.get("result").get("data_array")

        retriever_schema = self.retriever_cfg['retriever_schema']

        mapped_result = {}
        output_list = []

        if len(data_array) > 0:
            for data in data_array:
                for column, column_value in zip(columns, data):
                    mapped_result[column['name']] = column_value
                
                metadata = {'score': mapped_result['score']}
                doc = Document(
                    page_content = mapped_result[retriever_schema['text_column']],
                    metadata = metadata,
                    id = mapped_result[retriever_schema['primary_key']]
                )
                output_list.append(doc)

        return output_list
    
    @mlflow.trace(span_type=SpanType.RETRIEVER)
    def refined_search(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
        ):

        search_results = self.search(
            query_text=query_text,
            columns=columns,
            filters=filters,
            num_results=num_results,
        )

        parsed_results = self.parse_search_result(search_result=search_results)

        return parsed_results

In [0]:
client = VectorSearchWrapper(retriever_configs['retriever_1'])

# Normal search (docs don't render)
results = client.search("hello")

In [0]:
# Refined search (docs render)
refined_search = client.refined_search("hello")

## Multi Retriever

In [0]:
from typing import List, Dict, Optional, Any
from databricks_langchain import ChatDatabricks
import ast


class MultiRetrieverOrchestrator:
    def __init__(
        self,
        retriever_configs: List[Dict[str, Any]],
        llm_endpoint: str = "databricks-gpt-oss-120b",
    ):
        """
        Initialize with a list of retriever configurations.

        Each config should include:
            - endpoint_name
            - index_name
            - (optional) default_query_text
            - (optional) default_columns
            - (optional) default_filters
            - (optional) default_num_results

        :param llm_endpoint: The LLM endpoint used to generate queries.
        """
        self.retrievers = [
            VectorSearchWrapper(**config) for config in retriever_configs
        ]
        self.llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0)

    def generate_queries(self, query_text: str) -> List[str]:
        """
        Use the LLM to generate a list of queries, one per retriever.

        :param query_text: The original query from the user.
        :return: List of LLM-generated queries with length equal to number of retrievers.
        """
        num_queries = len(self.retrievers)

        response = self.llm.invoke(
            f"Generate {num_queries} variations of this query for vector search: '{query_text}'. Return the generated queries in a list. For example ['how are you?', 'how do you do?']"
        )

        text_items = [item['text'] for item in response.content if item.get('type') == 'text']
        if not text_items:
            raise ValueError("LLM response does not contain any items with type='text'")

        # Assume the first text item contains the list string
        text_str = text_items[0]

        # Convert string representation of list to Python list
        import ast
        try:
            queries = ast.literal_eval(text_str)
        except Exception as e:
            raise ValueError(f"Failed to parse LLM text as list: {e}")

        return queries

    def search_all(
        self,
        query_text: str,
        columns: Optional[List[str]] = None,
        filters: Optional[Dict[str, tuple]] = None,
        num_results: Optional[int] = None,
    ) -> Dict[str, Any]:
        """
        Run all retrievers in parallel using LLM-generated queries.

        :param query_text: Original query that will be fed to LLM.
        :return: Dict mapping retriever index -> results.
        """
        llm_queries = self.generate_queries(query_text)

        results = {}
        with ThreadPoolExecutor(max_workers=len(self.retrievers)) as executor:
            futures = {
                executor.submit(r.search, llm_query, columns, filters, num_results): i
                for i, (r, llm_query) in enumerate(
                    zip(self.retrievers, llm_queries), start=1
                )
            }
            for future in as_completed(futures):
                key = f"retriever_{futures[future]}"
                try:
                    results[key] = future.result()
                except Exception as e:
                    results[key] = f"Error: {e}"

        return results

In [0]:
retriever_configs = [
    {
        "endpoint_name": "one-env-shared-endpoint-13",
        "index_name": "global_fs.worldbank.chapter_index",
        "default_columns": ["chapter", "chapter_text"],
        "default_filters": {},
        "default_num_results": 2
    },
    {
        "endpoint_name": "one-env-shared-endpoint-13",
        "index_name": "global_fs.worldbank.chapter_index",
        "default_columns": ["chapter", "chapter_text"],
        "default_filters": {},
        "default_num_results": 5
    }
]


In [0]:
orchestrator = MultiRetrieverOrchestrator(retriever_configs)

In [0]:
import mlflow
mlflow.langchain.autolog()

results = orchestrator.search_all(
    query_text="hello"
)

In [0]:
results